# IFEST DAC 2026 — V12A LEGAL
## Structural Canonicalizer + Sparse Evidence Verifier + External Hard-Negative Augmentation

This notebook is deliberately **pretrained-free**. It does not load any pretrained model, checkpoint, embedding, FastText vector, language-model weight, or test pseudo-label.

Method contract:
- official train is the only competition-labelled development/evaluation source;
- external Indonesian news contributes **training examples only**;
- external article URL slug + its own body is treated as a positive title/body pair;
- hard negatives are generated deterministically from external training data only;
- official validation is 5-fold **component-safe** by exact normalized title/content graph;
- exact pair / unique canonical-content structural rules are fitted only on each reference fold;
- sparse verifier = word/character TF-IDF LinearSVM + dense factual-evidence LightGBM;
- baseline (official-only) and augmented model are evaluated with the same validation protocol;
- blend + threshold are transferred cross-fitted: each held fold's policy is selected only from the other four OOF folds;
- test is not read until training, validation, promotion gates, final policy and final models are frozen;
- no test inspection, pseudo-labeling, reconstruction, fitting, validation, tuning, or row-level manual override.

External source used by default: `hanifabdlh/context-1-indonesian-news-corpus` (150,466 Indonesian 2015 news articles). The original title is reconstructed from the article URL slug; rows failing strict title/body quality filters are discarded. This is raw data usage, not pretrained representation usage.

In [1]:
import os, re, gc, sys, json, math, time, random, hashlib, zipfile, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlparse, unquote

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import expit

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, confusion_matrix

# Install only ordinary libraries if the Kaggle image lacks them. No model weights are downloaded.
try:
    from lightgbm import LGBMClassifier
except Exception:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','lightgbm'])
    from lightgbm import LGBMClassifier
try:
    from datasets import load_dataset
except Exception:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','datasets'])
    from datasets import load_dataset

SEED = 20260911
N_SPLITS = 5
CPU = max(1, min(12, (os.cpu_count() or 4)-1))

# External-data budget: up to 140k genuine 2015 article/title positives + one hard negative each.
# This uses nearly the full 150,466-row source while remaining practical in Kaggle RAM/runtime.
EXT_POS_MAX = 140_000
EXT_VECTORIZER_FIT_MAX = 30_000
OFFICIAL_WEIGHT = 8.0

WORD_MAX_FEATURES = 24_000
CHAR_MAX_FEATURES = 32_000
SVC_C = 1.5

# Precommitted promotion gates from the agreed V12A design.
EXTERNAL_AUG_GAIN_GATE = 0.010
NEG_F1_GAIN_GATE = 0.020
STRUCT_LOO_ACC_GATE = 0.995
MIN_POSITIVE_FOLDS = 4

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORK/'ifest_v12a'
OUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
print('V12A legal pipeline')
print('CPU threads:',CPU,'CUDA visible:',os.environ.get('CUDA_VISIBLE_DEVICES','default'))
print('No pretrained model/embedding/checkpoint is used.')

V12A legal pipeline
CPU threads: 3 CUDA visible: default
No pretrained model/embedding/checkpoint is used.


In [2]:
# ---------------------------
# Official TRAIN discovery only
# ---------------------------
REQ={'id','title','content'}
def csv_cols(p):
    try: return set(pd.read_csv(p,nrows=2).columns)
    except Exception: return set()

roots=[Path('/kaggle/input'), Path('/mnt/data'), Path.cwd()]
all_csv=[]
for r in roots:
    if r.exists():
        all_csv += list(r.rglob('*.csv'))
# Only train.csv is opened here. Official test/sample are identified by filename only
# and remain unread until the final inference cell.
train_candidates=[]
test_named=[p for p in all_csv if p.name.lower()=='test.csv']
sample_named=[p for p in all_csv if p.name.lower()=='sample_submission.csv']
test_path=test_named[0] if test_named else None
sample_path=sample_named[0] if sample_named else None
for p in [x for x in all_csv if x.name.lower()=='train.csv']:
    cols=csv_cols(p)
    if REQ|{'label'} <= cols:
        try:
            n=sum(1 for _ in open(p,encoding='utf-8',errors='ignore'))-1
        except Exception: n=0
        if n>1000: train_candidates.append((n,p))
assert train_candidates, 'Official train.csv not found.'
train_path=max(train_candidates,key=lambda x:x[0])[1]
train=pd.read_csv(train_path)
train=train[['id','title','content','label']].copy()
train['title']=train.title.fillna('').astype(str)
train['content']=train.content.fillna('').astype(str)
train['label']=train.label.astype(int)
assert set(train.label.unique()) <= {0,1}
assert len(train)==14397, f'Unexpected official train size: {len(train)}'
print('Official train:',train_path,train.shape,train.label.value_counts().to_dict())
print('Official test content remains unread until final inference.')

Official train: /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv (14397, 4) {1: 12960, 0: 1437}
Official test content remains unread until final inference.


In [3]:
# ---------------------------
# Normalization, connected components, structural engine
# ---------------------------
_re_nonword=re.compile(r'[^0-9a-zA-Z]+')
def norm_text(s):
    return ' '.join(_re_nonword.sub(' ',str(s).lower()).split())

def toks(s): return norm_text(s).split()

train['nt']=train.title.map(norm_text)
train['nc']=train.content.map(norm_text)

class DSU:
    def __init__(self,n): self.p=list(range(n)); self.sz=[1]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a=self.find(a); b=self.find(b)
        if a==b:return
        if self.sz[a]<self.sz[b]: a,b=b,a
        self.p[b]=a; self.sz[a]+=self.sz[b]

def make_components(df):
    n=len(df); d=DSU(n); seen_t={}; seen_c={}
    for i,(t,c) in enumerate(zip(df.nt,df.nc)):
        if t in seen_t: d.union(i,seen_t[t])
        else: seen_t[t]=i
        if c in seen_c: d.union(i,seen_c[c])
        else: seen_c[c]=i
    roots=[d.find(i) for i in range(n)]
    mp={}; out=np.empty(n,dtype=np.int32)
    for i,r in enumerate(roots):
        if r not in mp: mp[r]=len(mp)
        out[i]=mp[r]
    return out

components=make_components(train)
train['component']=components
print('Connected components:',len(np.unique(components)))

sgkf=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
fold_id=np.full(len(train),-1,np.int8)
folds=[]
for f,(ref,held) in enumerate(sgkf.split(train,train.label,groups=components)):
    fold_id[held]=f; folds.append((ref,held))
    assert not (set(components[ref]) & set(components[held]))
    print(f'fold {f}: ref={len(ref)} held={len(held)} pos_ref={train.label.iloc[ref].mean():.4f} pos_held={train.label.iloc[held].mean():.4f}')
assert (fold_id>=0).all()

def build_struct_maps(df,idx):
    sub=df.iloc[idx]
    pair_tmp=defaultdict(set)
    content_pos=defaultdict(set)
    for t,c,y in zip(sub.nt,sub.nc,sub.label):
        pair_tmp[(t,c)].add(int(y))
        if int(y)==1: content_pos[c].add(t)
    pair_map={k:next(iter(v)) for k,v in pair_tmp.items() if len(v)==1}
    canonical={c:next(iter(ts)) for c,ts in content_pos.items() if len(ts)==1}
    return pair_map,canonical

def apply_struct(df,idx,pair_map,canonical):
    known=np.zeros(len(idx),bool); pred=np.zeros(len(idx),np.int8)
    for j,i in enumerate(idx):
        t=df.nt.iloc[i]; c=df.nc.iloc[i]
        if (t,c) in pair_map:
            known[j]=True; pred[j]=pair_map[(t,c)]
        elif c in canonical:
            known[j]=True; pred[j]=int(t==canonical[c])
    return known,pred

# Leave-one-row-out structural diagnostic (train only).
pair_counts=defaultdict(lambda:[0,0]); cont_pos_counts=defaultdict(Counter)
for t,c,y in zip(train.nt,train.nc,train.label):
    pair_counts[(t,c)][int(y)]+=1
    if int(y)==1: cont_pos_counts[c][t]+=1
loo_n=loo_ok=0
for t,c,y in zip(train.nt,train.nc,train.label):
    a=pair_counts[(t,c)].copy(); a[int(y)]-=1
    p=None
    if sum(a)>0 and (a[0]==0 or a[1]==0): p=1 if a[1]>0 else 0
    if p is None:
        ctr=cont_pos_counts[c].copy()
        if int(y)==1:
            ctr[t]-=1
            if ctr[t]<=0: del ctr[t]
        if len(ctr)==1:
            canon=next(iter(ctr)); p=int(t==canon)
    if p is not None:
        loo_n+=1; loo_ok+=int(p==int(y))
STRUCT_LOO_ACC=loo_ok/max(1,loo_n)
print('Structural LOO eligible:',loo_n,'accuracy:',round(STRUCT_LOO_ACC,6))

Connected components: 11526
fold 0: ref=11520 held=2877 pos_ref=0.8997 pos_held=0.9023
fold 1: ref=11540 held=2857 pos_ref=0.8986 pos_held=0.9065
fold 2: ref=11506 held=2891 pos_ref=0.8999 pos_held=0.9014
fold 3: ref=11480 held=2917 pos_ref=0.9006 pos_held=0.8985
fold 4: ref=11542 held=2855 pos_ref=0.9022 pos_held=0.8921
Structural LOO eligible: 4534 accuracy: 0.998897


In [4]:
# ---------------------------
# External 2015 Indonesian news: raw data only, no pretrained representation
# ---------------------------
EXT_DATASET='hanifabdlh/context-1-indonesian-news-corpus'

def title_from_url(u):
    try:
        path=unquote(urlparse(str(u)).path).rstrip('/')
        s=path.split('/')[-1]
    except Exception:
        s=str(u)
    s=re.sub(r'\.(?:html?|php)$','',s,flags=re.I)
    s=re.sub(r'[._\-]+',' ',s)
    s=re.sub(r'\s+',' ',s).strip()
    # Reject IDs / tiny or pathological slugs.
    tt=s.split()
    alpha=sum(any(ch.isalpha() for ch in x) for x in tt)
    if not (4<=len(tt)<=32) or alpha/max(1,len(tt))<0.65: return ''
    return s

def load_external():
    # First accept a user-attached Kaggle copy if present.
    local=[]
    for r in [Path('/kaggle/input')]:
        if r.exists(): local += list(r.rglob('*.parquet'))
    for p in local:
        try:
            q=pd.read_parquet(p)
            if {'source','context'} <= set(q.columns) and len(q)>20_000:
                print('External source: local parquet',p)
                return q[['source','context']].copy(), str(p)
        except Exception: pass
    print('External source: downloading raw dataset',EXT_DATASET)
    ds=load_dataset(EXT_DATASET,split='train')
    q=ds.to_pandas()[['source','context']].copy()
    return q, EXT_DATASET

raw_ext, ext_origin=load_external()
raw_ext['title']=raw_ext.source.map(title_from_url)
raw_ext['content']=raw_ext.context.fillna('').astype(str)
raw_ext=raw_ext[(raw_ext.title.str.len()>12) & (raw_ext.content.str.split().str.len()>=80)].copy()
raw_ext['nt']=raw_ext.title.map(norm_text); raw_ext['nc']=raw_ext.content.map(norm_text)
raw_ext=raw_ext[(raw_ext.nt.str.split().str.len().between(4,32)) & (raw_ext.nc.str.split().str.len()>=80)]
raw_ext=raw_ext.drop_duplicates(['nt','nc']).reset_index(drop=True)
if len(raw_ext)>EXT_POS_MAX:
    raw_ext=raw_ext.sample(EXT_POS_MAX,random_state=SEED).reset_index(drop=True)
assert len(raw_ext)>=20_000, f'External usable positives too small: {len(raw_ext)}'
print('External clean positive pairs:',len(raw_ext))

External source: downloading raw dataset hanifabdlh/context-1-indonesian-news-corpus


README.md:   0%|          | 0.00/437 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/125M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150466 [00:00<?, ? examples/s]

External clean positive pairs: 125642


In [5]:
# ---------------------------
# Deterministic hard-negative generation from EXTERNAL TRAINING DATA ONLY
# ---------------------------
NEGATIONS={'tidak','tak','bukan','belum','tanpa','jangan'}
STOP={'yang','dan','di','ke','dari','untuk','pada','dengan','ini','itu','jadi','ada','akan','atau','sebagai','karena','oleh','dalam','saat','setelah','hingga','soal','atas','baru','lebih','bagi','para'}

ext_titles=[x.split() for x in raw_ext.title.astype(str)]
ext_content_sets=[set(toks(x)) for x in raw_ext.content]
title_df=Counter()
number_pool=[]
for ts in ext_titles:
    low=[norm_text(x) for x in ts]
    title_df.update(set(x for x in low if x))
    number_pool += [x for x in low if re.fullmatch(r'\d+(?:[.,]\d+)?',x)]
replacement_pool=[w for w,c in title_df.items() if 5<=c<=5000 and len(w)>=3 and w not in STOP and not w.isdigit()]
if not number_pool: number_pool=['1','2','10','100','2020']

# Rare-token buckets create topical title swaps without pretrained embeddings.
inv=defaultdict(list)
for i,ts in enumerate(ext_titles):
    for w in set(norm_text(' '.join(ts)).split()):
        df=title_df.get(w,0)
        if 3<=df<=120 and len(w)>=4 and w not in STOP: inv[w].append(i)

def replace_once(title,content_set,rng,force_number=False):
    parts=title.split(); lows=[norm_text(x) for x in parts]
    if force_number:
        cand=[i for i,w in enumerate(lows) if re.fullmatch(r'\d+(?:[.,]\d+)?',w)]
        if cand:
            j=rng.choice(cand)
            for _ in range(40):
                r=rng.choice(number_pool)
                if r!=lows[j] and r not in content_set:
                    parts[j]=r; return ' '.join(parts),'number_substitution'
    # Prefer a supported informative token, then replace it by a token absent from the article.
    cand=[i for i,w in enumerate(lows) if w and w in content_set and len(w)>=3 and w not in STOP and w not in NEGATIONS and not w.isdigit()]
    cand=sorted(cand,key=lambda i:title_df.get(lows[i],10**9))
    if cand:
        j=cand[0] if rng.random()<0.7 else rng.choice(cand[:min(5,len(cand))])
        for _ in range(80):
            r=rng.choice(replacement_pool)
            if r!=lows[j] and r not in content_set:
                parts[j]=r; return ' '.join(parts),'token_substitution'
    return None,None

def corrupt_row(i):
    rng=random.Random(SEED*1000003+i)
    title=raw_ext.title.iloc[i]; cset=ext_content_sets[i]; low=toks(title)
    nums=[w for w in low if re.fullmatch(r'\d+(?:[.,]\d+)?',w)]
    neg=[w for w in low if w in NEGATIONS]
    u=rng.random()
    if nums and u<0.22:
        x,tp=replace_once(title,cset,rng,True)
        if x: return x,tp
    if neg and u<0.34:
        parts=title.split()
        for j,p in enumerate(parts):
            if norm_text(p) in NEGATIONS:
                return ' '.join(parts[:j]+parts[j+1:]),'negation_deletion'
    if u>0.82:
        rare=sorted([w for w in set(low) if w in inv],key=lambda w:title_df[w])
        for w in rare[:5]:
            choices=[j for j in inv[w] if j!=i]
            if choices:
                j=rng.choice(choices)
                other=raw_ext.title.iloc[j]
                if norm_text(other)!=norm_text(title): return other,'topical_title_swap'
    x,tp=replace_once(title,cset,rng,False)
    if x: return x,tp
    # deterministic final fallback: another title sharing no requirement
    j=(i*7919+104729)%len(raw_ext)
    if j==i: j=(j+1)%len(raw_ext)
    return raw_ext.title.iloc[j],'fallback_title_swap'

neg_titles=[]; neg_types=[]
for i in range(len(raw_ext)):
    x,tp=corrupt_row(i); neg_titles.append(x); neg_types.append(tp)

ext_pos=raw_ext[['title','content']].copy(); ext_pos['label']=1; ext_pos['neg_type']='positive'
ext_neg=raw_ext[['content']].copy(); ext_neg['title']=neg_titles; ext_neg['label']=0; ext_neg['neg_type']=neg_types

# A generated hard negative can rarely coincide with another genuine external
# (title, article) positive pair. Such rows are contradictory supervision, so
# quarantine them deterministically before any model sees them.
pos_nt=ext_pos.title.map(norm_text).to_numpy()
pos_nc=ext_pos.content.map(norm_text).to_numpy()
neg_nt=ext_neg.title.map(norm_text).to_numpy()
neg_nc=ext_neg.content.map(norm_text).to_numpy()
pos_keys=set(zip(pos_nt,pos_nc))
same_mask=(neg_nt==pos_nt)
conflict_mask=np.fromiter(((t,c) in pos_keys for t,c in zip(neg_nt,neg_nc)),dtype=bool,count=len(ext_neg))
dropped_same_as_source=int(same_mask.sum())
dropped_conflicting_negatives=int((conflict_mask & ~same_mask).sum())
drop_mask=same_mask | conflict_mask
if drop_mask.any():
    ext_neg=ext_neg.loc[~drop_mask].reset_index(drop=True)

external=pd.concat([ext_pos,ext_neg],ignore_index=True)
external['title']=external.title.fillna('').astype(str); external['content']=external.content.fillna('').astype(str)
external['nt']=external.title.map(norm_text); external['nc']=external.content.map(norm_text)
external=external[external.nt.str.len()>0].reset_index(drop=True)

# Strict post-filter integrity. Filtering collisions is a data-rigor bug fix;
# the V12A model, validation scheme, features, and promotion gates are unchanged.
post_neg_keys=list(zip(ext_neg.title.map(norm_text),ext_neg.content.map(norm_text)))
post_conflicting_neg=int(sum(k in pos_keys for k in post_neg_keys))
assert post_conflicting_neg==0, {'post_conflicting_neg':post_conflicting_neg}
SYNTHETIC_CHECK=bool(post_conflicting_neg==0 and external.label.nunique()==2 and len(ext_neg)>0)
assert SYNTHETIC_CHECK, {'post_conflicting_neg':post_conflicting_neg,'labels':external.label.unique().tolist()}
print('External training pairs:',external.shape,'labels:',external.label.value_counts().to_dict())
print('Synthetic negative types:',external.loc[external.label.eq(0),'neg_type'].value_counts().to_dict())
print('Dropped synthetic rows:',{'same_as_source':dropped_same_as_source,'conflicting_positive_pair':dropped_conflicting_negatives})
print('Synthetic artifact check: PASS')

External training pairs: (251281, 6) labels: {1: 125642, 0: 125639}
Synthetic negative types: {'token_substitution': 89850, 'topical_title_swap': 20437, 'number_substitution': 11363, 'negation_deletion': 3918, 'fallback_title_swap': 71}
Dropped synthetic rows: {'same_as_source': 0, 'conflicting_positive_pair': 3}
Synthetic artifact check: PASS


In [6]:
# ---------------------------
# Evidence features and evidence-focused pair text
# ---------------------------
MONTHS={'januari','februari','maret','april','mei','juni','juli','agustus','september','oktober','november','desember','jan','feb','mar','apr','jun','jul','agu','sep','okt','nov','des'}
UNITS={'persen','persentase','juta','miliar','triliun','ribu','orang','kasus','tahun','bulan','hari','jam','menit','km','kilometer','kg','ton','rupiah','rp','dolar','usd'}

# Evidence-window selection is deliberately unweighted and row-local.
# IDF weighting is supplied by the fold-fitted TF-IDF vectorizers below, so the
# official-only baseline does not consume any external corpus statistic.
def idf(w): return 1.0

def ngrams(x,n): return {' '.join(x[i:i+n]) for i in range(max(0,len(x)-n+1))}
def chargrams(s,n=3):
    s=norm_text(s).replace(' ','_')
    return {s[i:i+n] for i in range(max(0,len(s)-n+1))}
def numbers(s): return re.findall(r'\b\d+(?:[.,]\d+)?\b',str(s).lower())
def proper_tokens(s): return [x.lower() for x in re.findall(r'\b[A-Z][A-Za-z]{2,}\b',str(s))]

def best_window(title_tokens,content_tokens,win=100,stride=70):
    if not content_tokens: return [],0.0,0
    denom=sum(idf(w) for w in set(title_tokens)) or 1.0
    best=[]; bs=-1.; bi=0
    for st in range(0,max(1,len(content_tokens)),stride):
        ch=content_tokens[st:st+win]
        if not ch: break
        ss=set(ch); score=sum(idf(w) for w in set(title_tokens) if w in ss)/denom
        if score>bs: best,bs,bi=ch,score,st
        if st+win>=len(content_tokens): break
    return best,max(bs,0.0),bi

def row_features(title,content):
    tt=toks(title); ct=toks(content); ts=set(tt); cs=set(ct); lead=ct[:100]
    lset=set(lead); bw,bscore,bpos=best_window(tt,ct); bset=set(bw)
    denom=max(1,len(ts)); idenom=sum(idf(w) for w in ts) or 1.0
    big=ngrams(tt,2); tri=ngrams(tt,3); c_norm=' '.join(ct)
    num_t=numbers(title); num_c=set(numbers(content));
    neg_t=[w for w in tt if w in NEGATIONS]; neg_c=set(w for w in ct if w in NEGATIONS)
    mon_t=[w for w in tt if w in MONTHS]; mon_c=set(w for w in ct if w in MONTHS)
    prop=proper_tokens(title); prop_c=set(x.lower() for x in re.findall(r'\b[A-Z][A-Za-z]{2,}\b',str(content)))
    cg=chargrams(title,3); cc=chargrams(content,3)
    whole_cov=sum(w in cs for w in ts)/denom
    lead_cov=sum(w in lset for w in ts)/denom
    best_cov=sum(w in bset for w in ts)/denom
    idf_whole=sum(idf(w) for w in ts if w in cs)/idenom
    idf_lead=sum(idf(w) for w in ts if w in lset)/idenom
    vals={
        'title_words':len(tt),'content_words':len(ct),'log_content_words':math.log1p(len(ct)),
        'whole_token_cov':whole_cov,'lead_token_cov':lead_cov,'best_token_cov':best_cov,
        'idf_whole_cov':idf_whole,'idf_lead_cov':idf_lead,'idf_best_cov':bscore,
        'best_window_relpos':bpos/max(1,len(ct)),
        'bigram_cov':sum(g in c_norm for g in big)/max(1,len(big)),
        'trigram_cov':sum(g in c_norm for g in tri)/max(1,len(tri)),
        'char3_cov':len(cg&cc)/max(1,len(cg)),
        'num_count':len(num_t),'num_supported_frac':sum(x in num_c for x in num_t)/max(1,len(num_t)),
        'num_missing':sum(x not in num_c for x in num_t),
        'neg_count':len(neg_t),'neg_missing':sum(x not in neg_c for x in neg_t),
        'month_count':len(mon_t),'month_missing':sum(x not in mon_c for x in mon_t),
        'proper_count':len(prop),'proper_supported_frac':sum(x in prop_c for x in prop)/max(1,len(prop)),
        'title_content_len_ratio':len(tt)/max(1,len(ct)),
    }
    miss=[w for w in tt if w not in cs][:20]; pres=[w for w in tt if w in cs][:20]
    pair=('TITLE '+ ' '.join(tt) + ' LEAD ' + ' '.join(lead[:100]) +
          ' EVIDENCE ' + ' '.join(bw[:100]) +
          ' MISSING ' + ' '.join('miss_'+w for w in miss) +
          ' SUPPORTED ' + ' '.join('hit_'+w for w in pres))
    return vals,pair

def featurize(df,name):
    rows=[]; texts=[]; t0=time.time()
    for k,(t,c) in enumerate(zip(df.title,df.content)):
        v,s=row_features(t,c); rows.append(v); texts.append(s)
        if (k+1)%25000==0: print(name,'featurized',k+1,'/',len(df),'sec',round(time.time()-t0,1))
    X=pd.DataFrame(rows).astype(np.float32)
    return X,np.asarray(texts,dtype=object)

off_dense,off_text=featurize(train,'official')
ext_dense,ext_text=featurize(external,'external')
DENSE_COLS=off_dense.columns.tolist()
assert DENSE_COLS==ext_dense.columns.tolist()
print('Dense features:',DENSE_COLS)

external featurized 25000 / 251281 sec 16.6
external featurized 50000 / 251281 sec 33.3
external featurized 75000 / 251281 sec 50.1
external featurized 100000 / 251281 sec 66.7
external featurized 125000 / 251281 sec 83.5
external featurized 150000 / 251281 sec 100.3
external featurized 175000 / 251281 sec 117.0
external featurized 200000 / 251281 sec 133.8
external featurized 225000 / 251281 sec 150.3
external featurized 250000 / 251281 sec 167.1
Dense features: ['title_words', 'content_words', 'log_content_words', 'whole_token_cov', 'lead_token_cov', 'best_token_cov', 'idf_whole_cov', 'idf_lead_cov', 'idf_best_cov', 'best_window_relpos', 'bigram_cov', 'trigram_cov', 'char3_cov', 'num_count', 'num_supported_frac', 'num_missing', 'neg_count', 'neg_missing', 'month_count', 'month_missing', 'proper_count', 'proper_supported_frac', 'title_content_len_ratio']


In [7]:
# ---------------------------
# Sparse + dense training helpers
# ---------------------------
def official_sample_weight(y):
    y=np.asarray(y,int); n=len(y); c=np.bincount(y,minlength=2)
    cw=np.array([n/(2*max(1,c[0])), n/(2*max(1,c[1]))],dtype=np.float32)
    return OFFICIAL_WEIGHT*cw[y]

def make_vecs(fit_text):
    word=TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=2,max_df=.997,
                         max_features=WORD_MAX_FEATURES,sublinear_tf=True,dtype=np.float32)
    char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=3,max_df=.997,
                         max_features=CHAR_MAX_FEATURES,sublinear_tf=True,dtype=np.float32)
    word.fit(fit_text); char.fit(fit_text)
    return word,char

def sp_transform(word,char,text):
    return sparse.hstack([word.transform(text),char.transform(text)],format='csr',dtype=np.float32)

def new_lgb(seed):
    return LGBMClassifier(
        objective='binary',n_estimators=450,learning_rate=.03,num_leaves=31,
        min_child_samples=40,subsample=.90,colsample_bytree=.90,
        reg_alpha=.2,reg_lambda=2.0,random_state=seed,n_jobs=CPU,verbosity=-1
    )

def fit_one_fold(ref,held,use_external,seed):
    ref=np.asarray(ref); held=np.asarray(held)
    # Vectorizer vocabulary is learned from training data only.
    if use_external:
        rng=np.random.RandomState(seed)
        nfit=min(EXT_VECTORIZER_FIT_MAX,len(ext_text))
        ei=rng.choice(len(ext_text),size=nfit,replace=False)
        fit_text=np.concatenate([off_text[ref],ext_text[ei]])
    else:
        fit_text=off_text[ref]
    word,char=make_vecs(fit_text)
    Xr=sp_transform(word,char,off_text[ref]); Xh=sp_transform(word,char,off_text[held])
    yr=train.label.to_numpy()[ref]
    wr=official_sample_weight(yr)
    svc=LinearSVC(C=SVC_C,dual='auto',max_iter=5000,random_state=seed)
    lgb=new_lgb(seed)
    if use_external:
        Xe=sp_transform(word,char,ext_text)
        Xtrain=sparse.vstack([Xr,Xe],format='csr')
        ytrain=np.concatenate([yr,external.label.to_numpy(dtype=int)])
        wtrain=np.concatenate([wr,np.ones(len(external),np.float32)])
        svc.fit(Xtrain,ytrain,sample_weight=wtrain)
        Zd=np.vstack([off_dense.iloc[ref].to_numpy(),ext_dense.to_numpy()])
        lgb.fit(Zd,ytrain,sample_weight=wtrain)
        del Xe,Xtrain,Zd,ytrain,wtrain
    else:
        svc.fit(Xr,yr,sample_weight=wr)
        lgb.fit(off_dense.iloc[ref],yr,sample_weight=wr)
    ss=svc.decision_function(Xh).astype(np.float32)
    lp=lgb.predict_proba(off_dense.iloc[held])[:,1].astype(np.float32)
    del word,char,svc,lgb,Xr,Xh
    gc.collect()
    return ss,lp

def struct_for_fold(ref,held):
    pm,cm=build_struct_maps(train,ref)
    return apply_struct(train,held,pm,cm)

In [8]:
# ---------------------------
# 5-fold component-safe validation: official-only baseline vs external augmentation
# ---------------------------
y=train.label.to_numpy(dtype=int)
svm_base=np.zeros(len(train),np.float32); lgb_base=np.zeros(len(train),np.float32)
svm_aug=np.zeros(len(train),np.float32); lgb_aug=np.zeros(len(train),np.float32)
struct_known=np.zeros(len(train),bool); struct_pred=np.zeros(len(train),np.int8)
train_log=[]

for f,(ref,held) in enumerate(folds):
    print('\n===== fold',f,'=====')
    sk,sp=struct_for_fold(ref,held); struct_known[held]=sk; struct_pred[held]=sp
    t=time.time()
    sb,lb=fit_one_fold(ref,held,False,SEED+f)
    sa,la=fit_one_fold(ref,held,True,SEED+100+f)
    svm_base[held]=sb; lgb_base[held]=lb; svm_aug[held]=sa; lgb_aug[held]=la
    train_log.append({'fold':f,'seconds':time.time()-t,'struct_coverage':float(sk.mean())})
    print('fold',f,'complete sec',round(time.time()-t,1),'struct coverage',round(sk.mean(),4))

pd.DataFrame(train_log).to_csv(OUT/'training_log.csv',index=False)


===== fold 0 =====
fold 0 complete sec 1236.3 struct coverage 0.0

===== fold 1 =====
fold 1 complete sec 1227.5 struct coverage 0.0

===== fold 2 =====
fold 2 complete sec 1249.4 struct coverage 0.0

===== fold 3 =====
fold 3 complete sec 1246.9 struct coverage 0.0

===== fold 4 =====
fold 4 complete sec 1212.0 struct coverage 0.0


In [9]:
# ---------------------------
# Cross-fitted blend + threshold policy (held fold never selects its own policy)
# ---------------------------
ALPHAS=np.linspace(0,1,11)
THRESHOLDS=np.linspace(.05,.95,181)

def combine_score(svm,lgb,a): return a*expit(svm)+(1-a)*lgb

def make_pred(idx,svm,lgb,a,t):
    idx=np.asarray(idx)
    p=np.empty(len(idx),np.int8)
    k=struct_known[idx]; p[k]=struct_pred[idx][k]
    if (~k).any(): p[~k]=(combine_score(svm[idx][~k],lgb[idx][~k],a)>=t).astype(np.int8)
    return p

def select_policy(idx,svm,lgb):
    idx=np.asarray(idx); yy=y[idx]
    best=None
    for a in ALPHAS:
        s=combine_score(svm[idx],lgb[idx],a)
        k=struct_known[idx]
        for t in THRESHOLDS:
            p=np.where(k,struct_pred[idx],(s>=t).astype(np.int8))
            m=f1_score(yy,p,average='macro')
            key=(m,-abs(t-.5),-abs(a-.5))
            if best is None or key>best[0]: best=(key,float(a),float(t))
    return best[1],best[2]

def eval_variant(svm,lgb,name):
    final=np.zeros(len(train),np.int8); policies=[]; rows=[]
    for f,(ref,held) in enumerate(folds):
        # Policy is chosen using OOF predictions from the OTHER four folds only.
        policy_idx=np.where(fold_id!=f)[0]
        a,t=select_policy(policy_idx,svm,lgb)
        ph=make_pred(held,svm,lgb,a,t); final[held]=ph
        mf=f1_score(y[held],ph,average='macro'); per=f1_score(y[held],ph,labels=[0,1],average=None)
        policies.append((a,t))
        rows.append({'variant':name,'fold':f,'alpha':a,'threshold':t,'macro_f1':mf,'f1_0':per[0],'f1_1':per[1],
                     'struct_coverage':float(struct_known[held].mean())})
    return final,policies,pd.DataFrame(rows)

pred_base,pol_base,met_base=eval_variant(svm_base,lgb_base,'official_only')
pred_aug,pol_aug,met_aug=eval_variant(svm_aug,lgb_aug,'external_augmented')
metrics=pd.concat([met_base,met_aug],ignore_index=True)
metrics.to_csv(OUT/'fold_metrics.csv',index=False)

macro_base=f1_score(y,pred_base,average='macro'); macro_aug=f1_score(y,pred_aug,average='macro')
f0_base=f1_score(y,pred_base,labels=[0,1],average=None)[0]; f0_aug=f1_score(y,pred_aug,labels=[0,1],average=None)[0]
fold_gains=(met_aug.sort_values('fold').macro_f1.to_numpy()-met_base.sort_values('fold').macro_f1.to_numpy())
pos_folds=int((fold_gains>0).sum())
aug_gain=float(macro_aug-macro_base); neg_gain=float(f0_aug-f0_base)

PROMOTED=bool(
    pos_folds>=MIN_POSITIVE_FOLDS and
    aug_gain>=EXTERNAL_AUG_GAIN_GATE and
    neg_gain>=NEG_F1_GAIN_GATE and
    STRUCT_LOO_ACC>=STRUCT_LOO_ACC_GATE and
    SYNTHETIC_CHECK
)
summary={
    'macro_f1_official_only':float(macro_base),'macro_f1_external_augmented':float(macro_aug),
    'external_aug_gain':aug_gain,'f1_0_official_only':float(f0_base),'f1_0_external_augmented':float(f0_aug),
    'negative_f1_gain':neg_gain,'positive_folds':pos_folds,'fold_gains':fold_gains.tolist(),
    'struct_loo_eligible':int(loo_n),'struct_loo_accuracy':float(STRUCT_LOO_ACC),
    'synthetic_artifact_check':bool(SYNTHETIC_CHECK),
    'promoted':PROMOTED,
}
print(json.dumps(summary,indent=2))
print('Baseline confusion:\n',confusion_matrix(y,pred_base))
print('Augmented confusion:\n',confusion_matrix(y,pred_aug))

pd.DataFrame({'id':train.id,'label':y,'fold':fold_id,'struct_known':struct_known.astype(int),'struct_pred':struct_pred,
              'svm_base':svm_base,'lgb_base':lgb_base,'pred_base':pred_base,
              'svm_aug':svm_aug,'lgb_aug':lgb_aug,'pred_aug':pred_aug}).to_csv(OUT/'oof_v12a.csv',index=False)
with open(OUT/'validation_summary.json','w') as f: json.dump(summary,f,indent=2)

# Freeze final policy BEFORE test is loaded: median transferred held-fold policy.
def median_policy(pols):
    a=float(np.median([x[0] for x in pols])); t=float(np.median([x[1] for x in pols])); return a,t
FINAL_POLICY_BASE=median_policy(pol_base); FINAL_POLICY_AUG=median_policy(pol_aug)
print('Frozen final policies base/aug:',FINAL_POLICY_BASE,FINAL_POLICY_AUG)
print('PROMOTED =',PROMOTED)

{
  "macro_f1_official_only": 0.6232860960307528,
  "macro_f1_external_augmented": 0.616588656569731,
  "external_aug_gain": -0.006697439461021748,
  "f1_0_official_only": 0.3215905127310778,
  "f1_0_external_augmented": 0.31312458361092604,
  "negative_f1_gain": -0.00846592912015176,
  "positive_folds": 1,
  "fold_gains": [
    -0.015488270534353199,
    -0.014721912662437941,
    -0.01059083725500598,
    -0.013277864219487512,
    0.020506849388892134
  ],
  "struct_loo_eligible": 4534,
  "struct_loo_accuracy": 0.9988972209969122,
  "synthetic_artifact_check": true,
  "promoted": false
}
Baseline confusion:
 [[  461   976]
 [  969 11991]]
Augmented confusion:
 [[  470   967]
 [ 1095 11865]]
Frozen final policies base/aug: (0.5, 0.5099999999999999) (0.30000000000000004, 0.42999999999999994)
PROMOTED = False


In [10]:
# ---------------------------
# Final training on ALL official train (+ external for augmented). TEST still unread.
# ---------------------------
def fit_final(use_external,seed):
    if use_external:
        rng=np.random.RandomState(seed)
        nfit=min(EXT_VECTORIZER_FIT_MAX,len(ext_text)); ei=rng.choice(len(ext_text),nfit,replace=False)
        fit_text=np.concatenate([off_text,ext_text[ei]])
    else: fit_text=off_text
    word,char=make_vecs(fit_text)
    Xo=sp_transform(word,char,off_text); yo=y; wo=official_sample_weight(yo)
    svc=LinearSVC(C=SVC_C,dual='auto',max_iter=5000,random_state=seed)
    lgb=new_lgb(seed)
    if use_external:
        Xe=sp_transform(word,char,ext_text)
        Xa=sparse.vstack([Xo,Xe],format='csr')
        ya=np.concatenate([yo,external.label.to_numpy(dtype=int)])
        wa=np.concatenate([wo,np.ones(len(external),np.float32)])
        svc.fit(Xa,ya,sample_weight=wa)
        Zd=np.vstack([off_dense.to_numpy(),ext_dense.to_numpy()])
        lgb.fit(Zd,ya,sample_weight=wa)
        del Xe,Xa,ya,wa,Zd
    else:
        svc.fit(Xo,yo,sample_weight=wo); lgb.fit(off_dense,yo,sample_weight=wo)
    del Xo; gc.collect()
    return word,char,svc,lgb

print('Fitting final official-only model...')
final_base=fit_final(False,SEED+900)
print('Fitting final augmented model...')
final_aug=fit_final(True,SEED+901)
full_pair_map,full_canonical=build_struct_maps(train,np.arange(len(train)))
print('Final models and policies frozen. Official test may now be read for inference only.')

Fitting final official-only model...
Fitting final augmented model...
Final models and policies frozen. Official test may now be read for inference only.


In [11]:
# ---------------------------
# FINAL TEST INFERENCE ONLY — no fitting/tuning/inspection after this point
# ---------------------------
if test_path is None:
    # Filename-only discovery is still safe here; now content may be read because everything is frozen.
    named=[p for p in all_csv if p.name.lower()=='test.csv']
    test_path=named[0] if named else None
assert test_path is not None,'Official test.csv not found.'
test=pd.read_csv(test_path)[['id','title','content']].copy()
test['title']=test.title.fillna('').astype(str); test['content']=test.content.fillna('').astype(str)
test['nt']=test.title.map(norm_text); test['nc']=test.content.map(norm_text)
assert len(test)==3603, f'Unexpected official test size: {len(test)}'

test_dense,test_text=featurize(test,'test-inference')

def infer(model_pack,policy):
    word,char,svc,lgb=model_pack
    Xt=sp_transform(word,char,test_text)
    ss=svc.decision_function(Xt).astype(np.float32); lp=lgb.predict_proba(test_dense)[:,1].astype(np.float32)
    a,t=policy; score=combine_score(ss,lp,a)
    pred=(score>=t).astype(np.int8)
    # Frozen structural overlay learned from official train only.
    for i,(nt,nc) in enumerate(zip(test.nt,test.nc)):
        if (nt,nc) in full_pair_map: pred[i]=full_pair_map[(nt,nc)]
        elif nc in full_canonical: pred[i]=int(nt==full_canonical[nc])
    del Xt; gc.collect()
    return pred

p_base=infer(final_base,FINAL_POLICY_BASE)
p_aug=infer(final_aug,FINAL_POLICY_AUG)

# Preserve sample submission order if available; otherwise official test order.
def write_sub(pred,path):
    d=pd.DataFrame({'id':test.id.astype(str),'label':pred.astype(int)})
    if sample_path is not None:
        ss=pd.read_csv(sample_path)
        if set(['id','label'])<=set(ss.columns) and len(ss)==len(d):
            order=ss[['id']].copy(); order['id']=order.id.astype(str)
            d=order.merge(d,on='id',how='left',validate='one_to_one')
            assert d.label.notna().all(); d['label']=d.label.astype(int)
    d.to_csv(path,index=False)
    return d

sub_base=write_sub(p_base,OUT/'submission_v12a_official_only.csv')
sub_aug=write_sub(p_aug,OUT/'submission_v12a_candidate.csv')
if PROMOTED:
    sub_aug.to_csv(OUT/'submission_v12a.csv',index=False)
else:
    # Explicit name prevents accidental submission of a non-promoted experiment.
    sub_aug.to_csv(OUT/'submission_v12a_NOT_PROMOTED.csv',index=False)

# Do not print row-level predictions, changed rows, or test probabilities.
def sha(p):
    h=hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
print('Inference complete; no test-derived development follows.')
print('official-only submission sha256:',sha(OUT/'submission_v12a_official_only.csv'))
print('candidate submission sha256:',sha(OUT/'submission_v12a_candidate.csv'))
print('promoted submission written:',PROMOTED)

Inference complete; no test-derived development follows.
official-only submission sha256: b107c0e273aad4125a5e9e94e912acd466c1f49abe5619f3cfad82157b5aa8d5
candidate submission sha256: d1d79598ecdf6c3960e12330ba8e451f42e9df73acc228726584aa15a4e189c9
promoted submission written: False


In [12]:
# ---------------------------
# Reproducibility manifest + results bundle (no additional fitting)
# ---------------------------
manifest={
    'version':'V12A LEGAL','seed':SEED,'n_splits':N_SPLITS,
    'pretrained_models_or_embeddings':False,
    'external_origin':ext_origin,'external_dataset_id':EXT_DATASET,
    'external_positive_rows':int(len(raw_ext)),'external_total_training_pairs':int(len(external)),
    'official_train_rows':int(len(train)),'official_test_rows':int(len(test)),
    'word_max_features':WORD_MAX_FEATURES,'char_max_features':CHAR_MAX_FEATURES,'svc_c':SVC_C,
    'official_weight':OFFICIAL_WEIGHT,'promotion_gates':{
        'external_aug_gain':EXTERNAL_AUG_GAIN_GATE,'negative_f1_gain':NEG_F1_GAIN_GATE,
        'struct_loo_accuracy':STRUCT_LOO_ACC_GATE,'positive_folds':MIN_POSITIVE_FOLDS},
    'final_policy_official_only':{'alpha':FINAL_POLICY_BASE[0],'threshold':FINAL_POLICY_BASE[1]},
    'final_policy_external_augmented':{'alpha':FINAL_POLICY_AUG[0],'threshold':FINAL_POLICY_AUG[1]},
    'promoted':PROMOTED,
    'test_policy':'Official test was read only after validation, promotion decision, final policy, and final model fitting were frozen. Test predictions are used only for submission generation.'
}
with open(OUT/'manifest.json','w') as f: json.dump(manifest,f,indent=2)
with open(OUT/'config.json','w') as f:
    json.dump({k:v for k,v in manifest.items() if k not in ['test_policy']},f,indent=2)

zip_path=WORK/'results_v12a.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('Artifacts:',OUT)
print('Bundle:',zip_path,'size MB=',round(zip_path.stat().st_size/1e6,3))

Artifacts: /kaggle/working/ifest_v12a
Bundle: /kaggle/working/results_v12a.zip size MB= 0.363


### Output contract
- `submission_v12a_official_only.csv`: fully legal official-train-only sparse+structural baseline.
- `submission_v12a_candidate.csv`: fully legal external-augmented candidate.
- `submission_v12a.csv`: created **only** when every precommitted V12A promotion gate passes.
- `submission_v12a_NOT_PROMOTED.csv`: candidate retained for audit when gates fail; do not submit it as V12A.
- `oof_v12a.csv`, `fold_metrics.csv`, `validation_summary.json`, `manifest.json`: rigorous train-only evaluation evidence.
- `/kaggle/working/results_v12a.zip`: compact experiment bundle.

No code after the final inference cell may use official-test predictions for model fitting, thresholding, validation, selection, pseudo-labeling, reconstruction, or manual row correction.